# Лабораторная 1: Mushroom (OpenML)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## Константы и конфигурация

Все параметры эксперимента зафиксированы для воспроизводимости.

In [ ]:
# Константы проекта
OPENML_DATA_ID = 24
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET_COLUMN = "class"
RARE_THRESHOLD = 0.10
OUTPUT_COLUMN = "odor_rare_count"

# Пути
RAW_PATH = project_root / "data" / "raw" / "mushroom.csv"
PROCESSED_DIR = project_root / "data" / "processed"
MODEL_PATH = project_root / "models" / "mushroom.joblib"
REPORTS_DIR = project_root / "reports"

# Контрольная сумма ожидаемого датасета
EXPECTED_RAW_SHA256 = "6a2195026f61fbe898893fc40b5e64cab4c485aab611ac964b8974a0a1f01ce1"

# Параметры обучения
C_VALUES = [0.1, 1.0, 10.0]
CV_FOLDS = 5

# Расшифровка кодов запаха из описания Mushroom
ODOR_NAMES = {
    "a": "Миндальный",
    "l": "Анисовый",
    "c": "Креозотовый",
    "y": "Рыбный",
    "f": "Зловонный",
    "m": "Затхлый",
    "n": "Без запаха",
    "p": "Резкий",
    "s": "Пряный",
}

print(f"Проект: {project_root}")
print(f"OpenML ID: {OPENML_DATA_ID}, Random State: {RANDOM_STATE}")

## 1. Загрузка датасета

Загрузка OpenML Mushroom (ID 24) с проверкой контрольной суммы. Если файл уже существует и корректен, повторная загрузка не требуется.

In [ ]:
import hashlib
from sklearn.datasets import fetch_openml

def download_dataset():
    """Сохранить исходный CSV или проверить уже имеющуюся копию.
    
    Фиксированный OpenML ID исключает выбор другой версии по имени.
    SHA-256 проверяется до записи: неожиданный ответ не заменит данные.
    Существующий файл проверяется, но никогда не перезаписывается.
    """
    if RAW_PATH.exists():
        checksum = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
        if checksum != EXPECTED_RAW_SHA256:
            raise ValueError(
                "CSV не совпадает с зафиксированным снимком Mushroom."
            )
        print(f"Датасет уже существует и проверен: {RAW_PATH}")
        return RAW_PATH

    print(f"Загрузка датасета из OpenML (ID {OPENML_DATA_ID})...")
    dataset = fetch_openml(
        data_id=OPENML_DATA_ID,
        as_frame=True,
        parser="auto",
        data_home=str(project_root / ".cache" / "openml"),
    )
    
    # Явные кодировка и перевод строки обеспечивают одинаковый CSV
    content = dataset.frame.to_csv(index=False, lineterminator="\n").encode("utf-8")
    checksum = hashlib.sha256(content).hexdigest()
    
    if checksum != EXPECTED_RAW_SHA256:
        raise ValueError(
            "Ответ OpenML отличается от ожидаемого снимка Mushroom. "
            "CSV не сохранён; проверьте версии зависимостей и источник."
        )

    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = RAW_PATH.with_suffix(".csv.tmp")
    try:
        temporary_path.write_bytes(content)
        temporary_path.replace(RAW_PATH)
    finally:
        temporary_path.unlink(missing_ok=True)
    
    print(f"Датасет сохранён: {RAW_PATH}")
    return RAW_PATH

# Загрузить или проверить датасет
raw_path = download_dataset()
print(f"SHA-256: {hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()}")

## 2. Загрузка и валидация данных

Все 22 исходных признака категориальные; class — цель. Естественные пропуски stalk-root заполняются модой внутри Pipeline.

In [ ]:
import pandas as pd

def load_dataset():
    """Прочитать и проверить локальный CSV без сетевых запросов."""
    if not RAW_PATH.is_file():
        raise FileNotFoundError(
            f"Датасет не найден: {RAW_PATH}. Выполните ячейку загрузки."
        )

    checksum = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
    if checksum != EXPECTED_RAW_SHA256:
        raise ValueError("CSV не совпадает с зафиксированным снимком Mushroom.")

    # dtype=object сохраняет номинальный характер всех исходных признаков
    frame = pd.read_csv(RAW_PATH, dtype=object, na_values=["?"])
    
    if frame.shape != (8124, 23):
        raise ValueError("Для OpenML ID 24 ожидается 8124 строки и 23 колонки.")
    if set(frame[TARGET_COLUMN].dropna()) != {"e", "p"}:
        raise ValueError("Целевая переменная должна содержать классы e и p.")
    if frame[TARGET_COLUMN].isna().any():
        raise ValueError("Целевая переменная содержит пропуски.")
    
    return frame

frame = load_dataset()
print(f"Размер данных: {frame.shape}")
print(f"\nПропуски:\n{frame.isna().sum()[lambda s: s > 0]}")
print(f"\nРаспределение классов:\n{frame[TARGET_COLUMN].value_counts()}")
frame.head()

## 3. Разбиение на train/test

Train/test разделяются 80/20 с random_state=42 и стратификацией по классу. Это разбиение выполняется **до** обучения любых преобразований.


In [ ]:
from sklearn.model_selection import train_test_split

def split_dataset(frame):
    """Разделить данные 80/20, сохранив пропорции классов и исходные индексы."""
    features = frame.drop(columns=TARGET_COLUMN)
    target = frame[TARGET_COLUMN]
    return train_test_split(
        features,
        target,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=target,
    )

features_train, features_test, target_train, target_test = split_dataset(frame)

print(f"Train: {len(features_train)} строк")
print(f"Test: {len(features_test)} строк")
print(f"\nРаспределение классов в train:\n{target_train.value_counts()}")
print(f"\nРаспределение классов в test:\n{target_test.value_counts()}")


## 4. Анализ частот кодов запаха

Редкими считаем символы с долей **строго меньше 10%** от числа строк train. Список кодов заранее не задан и определяется данными.


In [ ]:
def describe_odor(features_train):
    """Посчитать все коды odor в train, от самого редкого к частому."""
    counts = features_train["odor"].value_counts(dropna=False)
    report = counts.rename_axis("code").reset_index(name="train_count")
    report["meaning"] = report["code"].map(ODOR_NAMES).fillna("Пропуск")
    report["train_percent"] = 100 * report["train_count"] / len(features_train)
    report["is_rare"] = report["code"].notna() & (
        report["train_percent"] < 100 * RARE_THRESHOLD
    )
    return report.sort_values(["train_count", "code"]).reset_index(drop=True)

odor_frequencies = describe_odor(features_train)
print(f"Порог редкости: {RARE_THRESHOLD:.0%} (строго меньше)\n")
odor_frequencies


## 5. Собственный трансформер RareOdorCounter

Наследует TransformerMixin и BaseEstimator. Добавляет числовой признак odor_rare_count: поскольку odor — одна буква, счётчик равен 0 либо 1. Неизвестная буква считается редкой; пропуск сохраняется как NaN для последующей импутации.


In [ ]:
import numpy as np
from numbers import Real
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class RareOdorCounter(TransformerMixin, BaseEstimator):
    """Считать редкие символы в однобуквенном признаке odor.
    
    Parameters
    ----------
    threshold : float, default=0.10
        Код редкий, если доля обучающих строк с ним строго меньше порога.
        Частота ровно 10% при пороге 0.10 не считается редкой.
    
    Notes
    -----
    Каждый odor в Mushroom содержит одну букву, поэтому счётчик равен
    0 или 1. Неизвестный код имеет частоту 0 и получает 1. Пропуск остаётся
    NaN для последующей импутации. Исходный odor сохраняется.
    Частоты обучаются только в fit, в том числе внутри каждого CV-фолда.
    """

    def __init__(self, threshold=RARE_THRESHOLD):
        self.threshold = threshold

    @staticmethod
    def _validate_frame(features):
        """Проверить формат входа, не меняя переданный DataFrame."""
        if not isinstance(features, pd.DataFrame):
            raise TypeError("RareOdorCounter ожидает pandas.DataFrame.")
        if not features.columns.is_unique or "odor" not in features:
            raise ValueError("Нужны уникальные колонки и признак odor.")
        if OUTPUT_COLUMN in features:
            raise ValueError(f"Признак {OUTPUT_COLUMN} уже существует.")
        valid = (
            features["odor"]
            .dropna()
            .map(lambda value: isinstance(value, str) and len(value) == 1)
        )
        if not valid.all():
            raise ValueError("odor должен содержать однобуквенные коды.")

    def fit(self, X, y=None):
        """Запомнить частоты train; цель y не используется."""
        self._validate_frame(X)
        if len(X) == 0:
            raise ValueError("Обучающая выборка не должна быть пустой.")
        if (
            isinstance(self.threshold, bool)
            or not isinstance(self.threshold, Real)
            or not 0 < self.threshold <= 1
        ):
            raise ValueError("threshold должен быть в интервале (0, 1].")
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        self.frequencies_ = X["odor"].value_counts().div(len(X)).to_dict()
        self.rare_symbols_ = tuple(
            sorted(
                symbol
                for symbol, frequency in self.frequencies_.items()
                if frequency < self.threshold
            )
        )
        return self

    def transform(self, X):
        """Добавить счётчик по частотам fit, сохранив индексы и исходный X."""
        check_is_fitted(self, "frequencies_")
        self._validate_frame(X)
        if not np.array_equal(X.columns, self.feature_names_in_):
            raise ValueError("Колонки и их порядок должны совпадать с fit.")
        result = X.copy()
        odor = X["odor"]
        frequencies = odor.map(self.frequencies_).fillna(0.0)
        counts = (frequencies < self.threshold).astype(float)
        result[OUTPUT_COLUMN] = counts.mask(odor.isna(), np.nan)
        return result

    def get_feature_names_out(self, input_features=None):
        """Вернуть имена исходных колонок и добавленного счётчика."""
        check_is_fitted(self, "feature_names_in_")
        if input_features is not None and not np.array_equal(
            input_features, self.feature_names_in_
        ):
            raise ValueError("input_features не совпадает с колонками fit.")
        return np.append(self.feature_names_in_, OUTPUT_COLUMN)

# Тест трансформера на train
transformer = RareOdorCounter().fit(features_train)
print(f"Редкие символы (частота < {RARE_THRESHOLD:.0%}): {transformer.rare_symbols_}")
print(f"\nЧастоты всех символов в train:")
for symbol, freq in sorted(transformer.frequencies_.items(), key=lambda x: x[1]):
    print(f"  {symbol}: {freq:.4f} ({'редкий' if freq < RARE_THRESHOLD else 'частый'})")


## 6. Построение Pipeline

**Числовая ветвь**: SimpleImputer(median) → StandardScaler для odor_rare_count  
**Категориальная ветвь**: SimpleImputer(most_frequent) → OneHotEncoder(handle_unknown="ignore")  
**Модель**: LogisticRegression с random_state=42


In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_pipeline():
    """Создать Pipeline; все статистики будут обучены только при fit."""
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, make_column_selector(dtype_include=np.number)),
        ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=np.number)),
    ], remainder="drop")
    
    return Pipeline([
        ("rare_odor", RareOdorCounter()),
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])

pipeline = build_pipeline()
print("Pipeline создан:")
pipeline


## 7. Обучение модели с подбором гиперпараметров

GridSearchCV выбирает C из [0.1, 1.0, 10.0] по ROC-AUC на пяти стратифицированных фолдах train. Лучший конвейер затем оценивается на отложенном test.

**Важно**: частоты редких символов пересчитываются для каждого фолда внутри GridSearchCV, исключая утечку данных.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from threadpoolctl import threadpool_limits

print("Запуск GridSearchCV...\n")
print(f"Параметры: C = {C_VALUES}")
print(f"Кросс-валидация: {CV_FOLDS} фолдов, стратифицированная")
print(f"Метрика отбора: ROC-AUC\n")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

search = GridSearchCV(
    estimator=build_pipeline(),
    param_grid={"classifier__C": C_VALUES},
    scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
    refit="roc_auc",
    cv=cv,
    n_jobs=1,
    error_score="raise",
)

# Один поток BLAS уменьшает различия округления между запусками
with threadpool_limits(limits=1):
    search.fit(features_train, target_train)

model = search.best_estimator_
best_params = search.best_params_
best_cv_score = search.best_score_

print(f"\nОбучение завершено!")
print(f"Лучшие параметры: {best_params}")
print(f"Лучший средний CV ROC-AUC: {best_cv_score:.8f}")

# Показать результаты всех комбинаций
cv_results = pd.DataFrame(search.cv_results_)[[
    "param_classifier__C",
    "mean_test_accuracy",
    "std_test_accuracy",
    "mean_test_roc_auc",
    "std_test_roc_auc",
    "rank_test_roc_auc",
]].rename(columns={"param_classifier__C": "C"})

cv_results


## 8. Оценка на тестовой выборке

Положительный класс — p (ядовитый). Вычисляем accuracy, ROC-AUC, F1 и матрицу ошибок.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

# Предсказания на test
with threadpool_limits(limits=1):
    predicted = model.predict(features_test)
    poison_index = list(model.classes_).index("p")
    poison_probability = model.predict_proba(features_test)[:, poison_index]

# Метрики
target_binary = (target_test == "p").astype(int)
matrix = confusion_matrix(target_test, predicted, labels=["e", "p"])

metrics = {
    "accuracy": float(accuracy_score(target_test, predicted)),
    "roc_auc": float(roc_auc_score(target_binary, poison_probability)),
    "f1_poisonous": float(f1_score(target_test, predicted, pos_label="p")),
    "test_rows": len(features_test),
    "errors": int(np.sum(predicted != target_test.to_numpy())),
    "confusion_matrix_labels": ["e", "p"],
    "confusion_matrix": matrix.tolist(),
}

print("=" * 60)
print("РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ")
print("=" * 60)
print(f"Accuracy:          {metrics['accuracy']:.8f}")
print(f"ROC-AUC:           {metrics['roc_auc']:.8f}")
print(f"F1 (poisonous):    {metrics['f1_poisonous']:.8f}")
print(f"Ошибок:            {metrics['errors']} / {metrics['test_rows']}")
print(f"\nМатрица ошибок [e, p]:")
print(matrix)
print(f"\n{classification_report(target_test, predicted, digits=4)}")


## 9. Сохранение результатов

Сохраняем модель, метрики, разбиения, предсказания и метаданные для воспроизводимости.


In [ ]:
import json
import joblib
import platform
from importlib.metadata import version

def write_json(path, value):
    """Записать JSON без NaN с единым форматированием."""
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

def save_split(features, target, name):
    """Сохранить split с исходными индексами для проверки разделения."""
    frame = features.copy()
    frame[TARGET_COLUMN] = target
    frame.to_csv(
        PROCESSED_DIR / f"{name}.csv",
        index=True,
        index_label="source_row",
        lineterminator="\n",
    )

# Создать директории
for directory in (REPORTS_DIR, PROCESSED_DIR, MODEL_PATH.parent):
    directory.mkdir(parents=True, exist_ok=True)

# Сохранить разбиения
save_split(features_train, target_train, "train")
save_split(features_test, target_test, "test")
print(f"Сохранены разбиения: {PROCESSED_DIR}/train.csv, test.csv")

# Сохранить метрики
write_json(REPORTS_DIR / "metrics.json", metrics)
print(f"Сохранены метрики: {REPORTS_DIR / 'metrics.json'}")

# Сохранить предсказания
pd.DataFrame({
    "source_row": features_test.index,
    "actual_class": target_test.to_numpy(),
    "predicted_class": predicted,
    "probability_poisonous": poison_probability,
}).to_csv(
    REPORTS_DIR / "test_predictions.csv",
    index=False,
    float_format="%.12f",
)
print(f"Сохранены предсказания: {REPORTS_DIR / 'test_predictions.csv'}")

# Сохранить результаты CV
cv_results.to_csv(
    REPORTS_DIR / "cv_results.csv",
    index=False,
    float_format="%.12f",
)
print(f"Сохранены результаты CV: {REPORTS_DIR / 'cv_results.csv'}")

# Сохранить частоты запахов
odor_frequencies.to_csv(
    REPORTS_DIR / "odor_frequencies.csv",
    index=False,
    float_format="%.6f",
)
print(f"Сохранены частоты: {REPORTS_DIR / 'odor_frequencies.csv'}")

# Получить информацию о трансформере
rare_counter = model.named_steps["rare_odor"]
feature_names = model[:-1].get_feature_names_out()

# Метаданные эксперимента
metadata = {
    "openml_id": OPENML_DATA_ID,
    "openml_version": 1,
    "raw_sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "train_rows": len(features_train),
    "test_rows": len(features_test),
    "train_class_counts": target_train.value_counts().to_dict(),
    "test_class_counts": target_test.value_counts().to_dict(),
    "missing_values": frame.isna().sum()[lambda s: s > 0].to_dict(),
    "input_feature_count": features_train.shape[1],
    "transformed_feature_count": len(feature_names),
    "rare_threshold": rare_counter.threshold,
    "rare_comparison": "strictly_less_than",
    "rare_symbols": list(rare_counter.rare_symbols_),
    "odor_train_frequencies": rare_counter.frequencies_,
    "cv_folds": CV_FOLDS,
    "C_candidates": C_VALUES,
    "best_params": best_params,
    "best_cv_roc_auc": float(best_cv_score),
    "positive_class": "p",
    "python": platform.python_version(),
    "versions": {
        name: version(name)
        for name in ("pandas", "numpy", "scipy", "scikit-learn", "joblib", "dvc")
    },
}
write_json(REPORTS_DIR / "run_metadata.json", metadata)
print(f"Сохранены метаданные: {REPORTS_DIR / 'run_metadata.json'}")

# Сохранить имена признаков
(REPORTS_DIR / "feature_names.json").write_text(
    json.dumps(feature_names.tolist(), indent=2) + "\n",
    encoding="utf-8",
)
print(f"Сохранены имена признаков: {REPORTS_DIR / 'feature_names.json'}")

# Сохранить модель
joblib.dump(model, MODEL_PATH, compress=3)
print(f"\nМодель сохранена: {MODEL_PATH}")

# Проверка загрузки
loaded_model = joblib.load(MODEL_PATH)
with threadpool_limits(limits=1):
    np.testing.assert_array_equal(
        loaded_model.predict(features_test), predicted
    )
    np.testing.assert_allclose(
        loaded_model.predict_proba(features_test)[:, poison_index],
        poison_probability,
        rtol=0,
        atol=0,
    )
print("Проверка: модель успешно загружена, предсказания идентичны")


## 10. Визуализация результатов

ROC-кривая и матрица ошибок на тестовой выборке.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import RocCurveDisplay

def save_evaluation_plots(target, probability, matrix, output_dir):
    """Построить ROC и матрицу ошибок отложенной тестовой выборки."""
    output_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", font="DejaVu Sans")
    
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    
    # ROC-кривая
    RocCurveDisplay.from_predictions(
        target, probability, ax=axes[0], name="Logistic regression"
    )
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title("ROC на тестовой выборке")
    
    # Матрица ошибок
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["e: съедобный", "p: ядовитый"],
        yticklabels=["e: съедобный", "p: ядовитый"],
        ax=axes[1],
    )
    axes[1].set_title("Матрица ошибок на тесте")
    axes[1].set_xlabel("Предсказанный класс")
    axes[1].set_ylabel("Истинный класс")
    
    figure.tight_layout()
    figure.savefig(output_dir / "evaluation.png", dpi=160)
    plt.close(figure)
    
    return output_dir / "evaluation.png"

figures_dir = REPORTS_DIR / "figures"
plot_path = save_evaluation_plots(target_binary, poison_probability, matrix, figures_dir)
print(f"График сохранён: {plot_path}")

# Отобразить в ноутбуке
from IPython.display import Image, display
display(Image(filename=str(plot_path)))


## 11. Итоговая сводка

Текстовое резюме эксперимента для быстрого просмотра.


In [ ]:
summary = (
    f"OpenML Mushroom (ID {OPENML_DATA_ID}), train={len(features_train)}, test={len(features_test)}\n"
    f"Rare odor: frequency < {rare_counter.threshold:.0%}\n"
    f"Rare symbols: {', '.join(rare_counter.rare_symbols_)}\n"
    f"Best parameters: {best_params}\n"
    f"Best mean CV ROC-AUC: {best_cv_score:.8f}\n"
    f"Test accuracy: {metrics['accuracy']:.8f}\n"
    f"Test ROC-AUC: {metrics['roc_auc']:.8f}\n"
    f"Test F1 (poisonous): {metrics['f1_poisonous']:.8f}\n"
    f"Errors: {metrics['errors']} / {len(features_test)}\n"
    f"Confusion matrix [e, p]: {matrix.tolist()}\n"
    f"Model saved: {MODEL_PATH}\n"
    f"Reload check: predictions and probabilities identical\n\n"
    + classification_report(target_test, predicted, digits=4)
)

(REPORTS_DIR / "training_summary.txt").write_text(summary, encoding="utf-8")
print(summary)


## 12. DVC: фиксация данных и модели

После выполнения всех ячеек выше, зафиксируйте данные и модель через DVC:

```bash
# Добавить исходные данные
dvc add data/raw/mushroom.csv

# Добавить разбиения и модель
dvc add data/processed/train.csv data/processed/test.csv models/mushroom.joblib

# Проверить статус
dvc status

# Добавить .dvc файлы в Git
git add data/raw/mushroom.csv.dvc data/raw/.gitignore
git add data/processed/train.csv.dvc data/processed/test.csv.dvc data/processed/.gitignore
git add models/mushroom.joblib.dvc models/.gitignore

# Зафиксировать в Git
git commit -m "Track lab1 data and model with DVC"
```

**Важно**: DVC-кэш локальный. Для совместной работы настройте удалённое хранилище (`dvc remote add`) и выполните `dvc push`.


## 13. Заполнение отчёта

Заполните `reports/report_template.md` реальными метриками и выводами. Добавьте скриншоты:

1. Структура проекта
2. Вывод `dvc status`
3. Результаты обучения и метрики
4. Интерфейс Jupyter (если используете)

Метрики доступны в:
- `reports/metrics.json` — основные метрики test
- `reports/run_metadata.json` — параметры, CV, редкие символы
- `reports/cv_results.csv` — результаты GridSearchCV
- `reports/training_summary.txt` — текстовая сводка

Сохраните скриншоты в `reports/screenshots/` и обновите шаблон отчёта.
